# SUFE VOS: Cutie Candidate and SAM2 Fusion

This notebook is the post-SAM3.1 mainline. It keeps code and full experiment outputs on Colab local disk, reads inputs from Drive, runs Cutie first-frame mask-conditioned VOS, and optionally fuses a completed SAM2 baseline with the Cutie candidate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import datetime, json, os, pathlib, re, shutil, subprocess, sys, zipfile

def clean_repo_url(value):
    text = str(value or '').strip()
    markdown = re.search(r'\]\((https?://[^)\s]+)\)', text)
    if markdown:
        text = markdown.group(1).strip()
    else:
        plain_url = re.search(r'https?://[^\s)>\]]+', text)
        if plain_url:
            text = plain_url.group(0).strip()
    if text.startswith('<') and text.endswith('>'):
        text = text[1:-1].strip()
    return text.strip("`'\"")

def run_streamed(cmd, label, env=None):
    child_env = os.environ.copy() if env is None else dict(env)
    child_env.setdefault('PYTHONUNBUFFERED', '1')
    child_env.setdefault('PYTHONIOENCODING', 'utf-8')
    print(f'Running {label}:', ' '.join(str(part) for part in cmd), flush=True)
    process = subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=None, stderr=None, env=child_env)
    return subprocess.CompletedProcess(cmd, process.wait())

def run_checked(cmd, label, env=None):
    result = run_streamed(cmd, label, env=env)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}. Check the streamed output and experiment JSON.')
    return result

def read_json(path):
    path = pathlib.Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def show_exp(exp_dir, limit=8000):
    exp_dir = pathlib.Path(exp_dir)
    print('\nEXP:', exp_dir)
    print('exists:', exp_dir.exists())
    print('submission:', (exp_dir / 'submission.zip').exists())
    print('mask_count:', len(list((exp_dir / 'masks').glob('**/*.png'))) if (exp_dir / 'masks').exists() else 0)
    for rel in ['sanity_check.json', 'run_manifest.json', 'logs/per_video_status.json']:
        path = exp_dir / rel
        print('\n---', rel, '---')
        if path.exists():
            print(path.read_text(encoding='utf-8')[:limit])
        else:
            print('MISSING')

DRIVE_ROOT = pathlib.Path(os.environ.get('SUFE_DRIVE_ROOT', '/content/drive/MyDrive')).expanduser().resolve()
REPO_URL = clean_repo_url(os.environ.get('SUFE_REPO_URL', 'https://github.com/yyj11266/SUFE_DL_FinalProject_Vision.git'))
PROJECT_ROOT = pathlib.Path(os.environ.get('SUFE_PROJECT_ROOT', '/content/sufe_vos_leaderboard')).expanduser().resolve()
DATA_ROOT = pathlib.Path(os.environ.get('SUFE_DATA_ROOT', '/content/sufe_data/video_dataset')).expanduser().resolve()
DATA_ZIP = pathlib.Path(os.environ.get('SUFE_DATA_ZIP', str(DRIVE_ROOT / 'sufe_vos_inputs/video_dataset.zip'))).expanduser().resolve()
SAMPLE_SUBMISSION = pathlib.Path(os.environ.get('SUFE_SAMPLE_SUBMISSION', str(DRIVE_ROOT / 'sufe_vos_inputs/sample_submission.zip'))).expanduser().resolve()
OUTPUT_ROOT = pathlib.Path(os.environ.get('SUFE_OUTPUTS_ROOT', '/content/sufe_runs')).expanduser().resolve()
REVIEW_ROOT = pathlib.Path(os.environ.get('SUFE_REVIEW_ROOT', str(DRIVE_ROOT / 'sufe_vos_review/runs'))).expanduser().resolve()
ARCHIVE_ROOT = pathlib.Path(os.environ.get('SUFE_ARCHIVE_ROOT', str(DRIVE_ROOT / 'sufe_vos_archives'))).expanduser().resolve()
CUTIE_REPO = pathlib.Path(os.environ.get('CUTIE_REPO_DIR', '/content/hkchengrex_cutie')).expanduser().resolve()

if (PROJECT_ROOT / '.git').exists():
    run_checked(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], 'git pull')
elif not PROJECT_ROOT.exists() and REPO_URL:
    run_checked(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], 'git clone')
elif PROJECT_ROOT.exists() and not any(PROJECT_ROOT.iterdir()) and REPO_URL:
    PROJECT_ROOT.rmdir()
    run_checked(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], 'git clone')
elif PROJECT_ROOT.exists() and not (PROJECT_ROOT / 'src' / 'data' / 'inspect_sufe.py').exists():
    raise RuntimeError(f'PROJECT_ROOT exists but is not this repo: {PROJECT_ROOT}. Remove it or set SUFE_PROJECT_ROOT.')

sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('HEAD:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('DATA_ZIP:', DATA_ZIP, 'exists=', DATA_ZIP.exists())
print('SAMPLE_SUBMISSION:', SAMPLE_SUBMISSION, 'exists=', SAMPLE_SUBMISSION.exists())
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('CUTIE_REPO:', CUTIE_REPO)


## Prepare Inputs

This cell extracts `video_dataset.zip` from Drive only when the local dataset directory is empty, or when `FORCE_EXTRACT_DATA=1` is set in the environment.

In [ ]:
FORCE_EXTRACT = os.environ.get('FORCE_EXTRACT_DATA', '0').strip().lower() in {'1', 'true', 'yes'}
if not SAMPLE_SUBMISSION.exists():
    raise RuntimeError(f'Missing sample submission: {SAMPLE_SUBMISSION}')
if not DATA_ZIP.exists():
    raise RuntimeError(f'Missing dataset zip: {DATA_ZIP}')

current_files = sum(1 for path in DATA_ROOT.rglob('*') if path.is_file()) if DATA_ROOT.exists() else 0
if FORCE_EXTRACT or current_files == 0:
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset zip to', DATA_ROOT)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        archive.extractall(DATA_ROOT)
else:
    print('Using existing extracted dataset:', DATA_ROOT)

from src.data.inspect_sufe import inspect_dataset
info = inspect_dataset(DATA_ROOT)
print('files:', info.scan['num_files'])
print('videos:', len(info.videos))
print('prompted videos:', info.prompt_format['num_prompted_videos'])
print('first videos:', [video.video_id for video in info.videos[:8]])
print('top-level:', info.scan['top_level_entries'][:20])
if info.scan['num_files'] <= 0 or not info.videos:
    raise RuntimeError(f'Dataset is still empty or unrecognized at {DATA_ROOT}')
if len(info.videos) != 25:
    print('WARNING: expected 25 SUFE videos, inspect_dataset found', len(info.videos))


## Cutie Smoke Run

Run the fixed six-video, 20-frame smoke subset first. Do not start the full run until this cell completes successfully.

In [ ]:
CUTIE_MAX_INTERNAL_SIZE = int(os.environ.get('CUTIE_MAX_INTERNAL_SIZE', '720'))
CUTIE_SMOKE_EXP = os.environ.get('CUTIE_SMOKE_EXP', 'cutie_smoke_720_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
smoke_dir = OUTPUT_ROOT / CUTIE_SMOKE_EXP
smoke_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_cutie_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_SUBMISSION),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', CUTIE_SMOKE_EXP,
    '--cutie-repo-dir', str(CUTIE_REPO),
    '--max-internal-size', str(CUTIE_MAX_INTERNAL_SIZE),
    '--smoke',
    '--max-frames', os.environ.get('CUTIE_SMOKE_MAX_FRAMES', '20'),
    '--save-overlays', os.environ.get('CUTIE_SAVE_OVERLAYS', 'sample'),
]
if os.environ.get('CUTIE_INSTALL', '1').strip().lower() not in {'0', 'false', 'no'}:
    smoke_cmd.append('--install-cutie')
try:
    run_checked(smoke_cmd, 'cutie-smoke')
except Exception:
    show_exp(smoke_dir)
    raise
show_exp(smoke_dir)

status = read_json(smoke_dir / 'logs/per_video_status.json') or {}
summary = status.get('summary', {})
if summary.get('status') != 'done':
    raise RuntimeError(f'Cutie smoke did not finish cleanly: {summary}')
if len(list((smoke_dir / 'masks').glob('**/*.png'))) <= 0:
    raise RuntimeError('Cutie smoke produced no masks')
print('CUTIE_SMOKE_EXP =', CUTIE_SMOKE_EXP)


## Cutie Full Candidate

Run this only after the smoke run looks acceptable. The output remains in `/content/sufe_runs/EXP_ID`; Drive receives only the compact review bundle in the later publish cell.

In [ ]:
CUTIE_FULL_EXP = os.environ.get('CUTIE_FULL_EXP', 'cutie_full_720')
full_dir = OUTPUT_ROOT / CUTIE_FULL_EXP
full_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_cutie_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_SUBMISSION),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', CUTIE_FULL_EXP,
    '--cutie-repo-dir', str(CUTIE_REPO),
    '--max-internal-size', str(CUTIE_MAX_INTERNAL_SIZE),
    '--save-overlays', os.environ.get('CUTIE_SAVE_OVERLAYS', 'sample'),
    '--skip-existing',
    '--make-submission',
]
try:
    run_checked(full_cmd, 'cutie-full')
except Exception:
    show_exp(full_dir)
    raise
show_exp(full_dir)

sanity = read_json(full_dir / 'sanity_check.json') or {}
if not sanity.get('passed'):
    raise RuntimeError(f'Cutie full sanity check failed: {sanity}')
if sanity.get('num_actual_masks') not in {None, 2015}:
    raise RuntimeError(f'Unexpected full mask count in sanity_check.json: {sanity}')
print('CUTIE_FULL_EXP =', CUTIE_FULL_EXP)


## Publish Cutie Review Bundle

Publish only the compact review bundle to Drive. The full `masks/` tree stays on Colab local disk.

In [ ]:
cutie_publish_dir = REVIEW_ROOT / CUTIE_FULL_EXP
publish_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/publish_run_for_codex.py'),
    '--exp-dir', str(OUTPUT_ROOT / CUTIE_FULL_EXP),
    '--publish-dir', str(cutie_publish_dir),
    '--data-root', str(DATA_ROOT),
    '--archive-dir', str(ARCHIVE_ROOT),
    '--replace',
]
if os.environ.get('SUFE_MAKE_FULL_ARCHIVE', '0').strip().lower() in {'1', 'true', 'yes'}:
    publish_cmd.append('--make-full-archive')
run_checked(publish_cmd, 'publish-cutie-review')
print('Cutie review bundle:', cutie_publish_dir)


## SAM2 + Cutie Conservative Fusion

Run this only after you have a completed SAM2 baseline experiment. Set `SAM2_BASELINE_EXP` to either an experiment id under `/content/sufe_runs` or an absolute path.

In [ ]:
SAM2_BASELINE_EXP = os.environ.get('SAM2_BASELINE_EXP', '').strip()
if not SAM2_BASELINE_EXP:
    raise RuntimeError('Set os.environ["SAM2_BASELINE_EXP"] to the SAM2 baseline exp id or absolute path before running fusion.')
baseline_exp = pathlib.Path(SAM2_BASELINE_EXP).expanduser()
if not baseline_exp.is_absolute():
    baseline_exp = OUTPUT_ROOT / baseline_exp
if not baseline_exp.exists():
    raise RuntimeError(f'SAM2 baseline experiment does not exist: {baseline_exp}')

FUSION_EXP = os.environ.get('FUSION_EXP', 'sam2_cutie_fusion_720')
fusion_dir = OUTPUT_ROOT / FUSION_EXP
fusion_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_sam2_cutie_fusion.py'),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_SUBMISSION),
    '--baseline-exp', str(baseline_exp),
    '--cutie-exp', str(OUTPUT_ROOT / CUTIE_FULL_EXP),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', FUSION_EXP,
    '--min-cutie-area', os.environ.get('FUSION_MIN_CUTIE_AREA', '16'),
    '--min-sam2-iou', os.environ.get('FUSION_MIN_SAM2_IOU', '0.50'),
    '--min-temporal-iou', os.environ.get('FUSION_MIN_TEMPORAL_IOU', '0.35'),
    '--min-area-ratio', os.environ.get('FUSION_MIN_AREA_RATIO', '0.20'),
    '--max-area-ratio', os.environ.get('FUSION_MAX_AREA_RATIO', '3.50'),
    '--skip-existing',
    '--make-submission',
]
try:
    run_checked(fusion_cmd, 'sam2-cutie-fusion')
except Exception:
    show_exp(fusion_dir)
    raise
show_exp(fusion_dir)

sanity = read_json(fusion_dir / 'sanity_check.json') or {}
if not sanity.get('passed'):
    raise RuntimeError(f'Fusion sanity check failed: {sanity}')
print('FUSION_EXP =', FUSION_EXP)


## Publish Fusion Review Bundle

In [ ]:
fusion_publish_dir = REVIEW_ROOT / FUSION_EXP
publish_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/publish_run_for_codex.py'),
    '--exp-dir', str(OUTPUT_ROOT / FUSION_EXP),
    '--publish-dir', str(fusion_publish_dir),
    '--data-root', str(DATA_ROOT),
    '--archive-dir', str(ARCHIVE_ROOT),
    '--replace',
]
if os.environ.get('SUFE_MAKE_FULL_ARCHIVE', '0').strip().lower() in {'1', 'true', 'yes'}:
    publish_cmd.append('--make-full-archive')
run_checked(publish_cmd, 'publish-fusion-review')
print('Fusion review bundle:', fusion_publish_dir)
